---
title: "08. The environment contract"
description: "The variables, schemas, trigger APIs, and promotion rules that keep the Docker Compose POC and the Azure deployment behaviorally aligned while their control-plane adapters remain separate."
categories: []
---

This chapter defines the contract that lets every Part I feature survive the transfer to Azure: the environment variables each image may read, who owns the results schema, equivalent trigger inputs, how promotion works, and the behavioral checks both validation adapters must preserve. Compose remains the complete local feature/demo environment; ACA supplies the cloud control plane and managed-identity adapter.


## Why a contract

The platform exists in two deployment phases, not two implementations. Part I
runs the complete feature/demo environment on one machine with Docker Compose:
database, object storage, MLflow, training, batch scoring, serving, dashboard,
and the local runner. Part II moves the same workload images and entrypoints to
Azure Container Apps (ACA), where long-lived HTTP services run as Apps,
run-to-finish workloads run as Jobs, and managed identities replace stored
passwords.

The control-plane mechanisms are allowed to differ. The local runner accepts
HTTP requests and starts subprocesses; the cloud dashboard starts ACA Job
executions. What must not differ is the workload contract: environment fallbacks,
results schema and statuses, model version identity, health/readiness routes,
prediction shape, and the evidence used to decide that a run succeeded.

The design rule is therefore: maximize source reuse, isolate deployment adapters,
and compare behavior rather than forcing local code to impersonate Azure APIs.


## The environment-variable contract

Application images contain code only: no models, no credentials, no
environment-specific values. All configuration arrives through environment
variables set by the surrounding platform, so the contract has a mechanical
definition: grep os.environ under projects/ml-platform/src/ to enumerate every
variable any image can read.

The columns: Consumed by names the image(s) reading the variable; Compose value
is what Part I sets in demo/docker-compose.yml or at trigger time; Azure source
is where Part II gets the same value from a Terraform-managed App/Job definition,
a managed identity, or runtime injection.

| Variable | Consumed by | Purpose | Compose value | Azure source |
|---|---|---|---|---|
| MLFLOW_TRACKING_URI | train, batch, serving, runner, dashboard | Self-hosted MLflow URL | http://mlflow:5000 | MLflow ACA App FQDN |
| MODEL_NAME | batch, serving, runner | Registered model name | wine-quality | Job/App definitions |
| MODEL_VERSION | serving, runner | Exact pinned version | 1 | Serving App; changed at promotion |
| DATA_SOURCE | batch | Scheduled batch input | Public wine CSV URL | Batch Job template |
| BATCH_CHUNK_SIZE | batch | Rows per scoring chunk | 100 default | 100 default |
| PORT | serving, dashboard | HTTP listen port | 8080 | App target port |
| PGHOST | jobs, runner, dashboard | PostgreSQL host | postgres | Azure PostgreSQL FQDN |
| PGPORT | jobs, dashboard | PostgreSQL port | 5432 | 5432 default |
| PGUSER | jobs, dashboard | Database role | mlplatform | Workload identity role |
| PGPASSWORD | jobs, dashboard | Local database password | demo password | Never set; Entra token becomes password |
| PGSSLMODE | jobs, dashboard | TLS policy | disable | require |
| RESULTS_DB | jobs, dashboard | Operational-state database | results | results |
| TRIGGERED_BY | jobs via record_run | Audit attribution | demo-bootstrap or runner value | Dashboard caller or schedule |
| RESULTS_RUN_ID | batch, jobs | Correlates local execution and result row | Runner execution id | UUID fallback |
| IMAGE_DIGEST | train and LLM entrypoints | Image lineage tag | demo-local | ACA digest |
| LLM_MODEL_NAME | LLM evaluator | Candidate model name | llm-app | LLM Job template |
| LLM_MODEL_VERSION | LLM evaluator | Candidate version | 1 | LLM Job template |
| LLM_EVAL_DATASET | LLM evaluator | JSONL evaluation fixture | /app/llm-eval.jsonl | LLM Job template |
| MODEL_API_KEY | LLM pyfunc | Local external-model credential | Optional env value | Not injected; Key Vault lookup |
| MODEL_API_KEY_SECRET | LLM pyfunc | Key Vault secret name | Not needed locally | LLM Job template |
| KEY_VAULT_URL | LLM pyfunc | Runtime secret store | Not needed locally | Foundation output |
| MLFLOW_UI_URL | dashboard | Browser-reachable MLflow link | localhost published port | Optional browser URL |
| GRAFANA_URL | dashboard | Grafana deep link | empty | Managed Grafana URL |
| TRIGGER_BACKEND | dashboard | Local runner vs ACA API | local | aca |
| RUNNER_URL | dashboard | Local execution-plane URL | http://runner:8090 | Not set |
| AZURE_SUBSCRIPTION_ID, AZURE_RESOURCE_GROUP | dashboard | ACA Jobs API address | unset | Terraform/deployment values |
| ACA_ENV_NAME | dashboard | Reserved ACA environment name | unset | Read today but currently unused |

Some variables exist on one side only. RUNNER_URL and TRIGGER_BACKEND select the
local control-plane adapter; RESULTS_RUN_ID is local correlation state; Key Vault
variables are cloud credential-delivery inputs. Those asymmetries are part of
the contract rather than accidental drift.

Change rule: adding a variable means updating the reader under src/, both
environments' configuration surfaces (Compose or Terraform), and this table.


## Results schema ownership

One table, `results`, in a dedicated PostgreSQL database is the canonical
operational state of every workflow (chapter 04): `id`, `parent_id`, `name`,
`status`, `output` (JSONB), `error`, `attempts`, `triggered_by`, and timestamps.
Its DDL lives in exactly one file,
`src/ml_platform/results/schema.sql`, written with `IF NOT EXISTS` guards so it
is safe to reapply forever.

Each environment applies it through its own bootstrap mechanism:

| Environment | Mechanism |
|---|---|
| Docker Compose | The official `postgres` image runs every script mounted in `/docker-entrypoint-initdb.d` on first initialization. `demo/postgres/init/01-create-results.sql` is mounted there; it creates the `results` database and then applies the same guarded DDL. This copy must be kept in lockstep with `schema.sql`. |
| Azure | After Terraform provisions the server, `deploy/deploy.sh` first runs `infra/grants.sql`, mapping each managed identity to a least-privilege Postgres role (train/batch read-write, dashboard read-only), then applies the canonical `schema.sql` verbatim. |

The status vocabulary is a fixed enum written only through
`ml_platform.results.store.mark`: `PENDING`, `STARTED`, `SUCCESS`, `RETRY`,
`FAILURE`, `REVOKED`. `RETRY` marks a transient failure eligible for
re-execution, `FAILURE` is permanent, and `REVOKED` marks a cancelled run.
Batch fan-out is expressed inside this one table: a parent row per batch plus
one child row per chunk, the parent finalized as `SUCCESS` only when no child
ended in `FAILURE`.

**Ownership rule:** schema changes land in `schema.sql` first and propagate to
the Compose init copy in the same change. Everything that reads the table
outside Python (the dashboard's SQL, Grafana panels) depends on these exact
column names, so there are no second definitions of "what a run looks like".


## Trigger API equivalence

The dashboard is a **catalog + launcher**, never an executor. It holds no Docker
socket and no broad cloud credentials; every launch is a request against an
**execution-plane endpoint**. The two backends implement the same shape:

- **Local:** `demo/runner/` is a small FastAPI service. Its
  `POST /api/jobs/{name}/run` accepts `{"triggered_by": ..., "parameters": {...}}`
  and runs the *same* `train.py` / `score.py` entrypoints as a subprocess,
  recording the same results rows. It deliberately does not integrate with the
  Docker daemon, which keeps the POC safe on a laptop while preserving the
  production interaction shape.
- **Azure:** the ACA Jobs API (`az containerapp job start`, or the SDK's
  `begin_start`) starts an execution of the pinned image; `triggered_by` rides
  along as an environment override.

At the control-plane level the runner mirrors the ACA Jobs surface, endpoint
for endpoint:

| Local runner (`demo/runner/app.py`) | Azure Container Apps Jobs |
|---|---|
| `POST /api/jobs/{name}/run` | `az containerapp job start` / SDK `begin_start` |
| `GET /api/jobs/{execution}` | `az containerapp job execution show` |
| `GET /api/jobs` | `az containerapp job execution list` |
| Execution id `local-train-1a2b3c4d`, returned by the trigger call | Execution name assigned by ACA, returned by the trigger call |
| Provenance rides as env vars: `TRIGGERED_BY`, `RESULTS_RUN_ID` | Same variables passed as container overrides at start |

Read the next table from the dashboard user's seat instead:

| Operation | Dashboard route | Local execution plane | Azure (ACA) |
|---|---|---|---|
| Trigger training | `POST /api/runs/train/trigger` | `POST {RUNNER_URL}/api/jobs/train/run` | `az containerapp job start --name train` |
| Trigger batch scoring | `POST /api/runs/batch/trigger` | `POST {RUNNER_URL}/api/jobs/batch/run` | same against the `batch` Job |
| Parameters | Pydantic models with documented defaults | per-job allow-list; unknown key rejected with 422 | rejected in this POC until Job argument overrides are wired in |
| Poll result | `GET /api/results/{result_id}` | same; `result_id` equals the execution id (the runner sets `RESULTS_RUN_ID`) | ACA execution API for orchestration state, results DB row for the outcome |
| Execution status + logs | `GET /api/executions/{execution}` | runner returns exit code plus the last 4000 characters of merged output | `az containerapp job logs show` / Log Analytics |

The parameter allow-lists come from the runner (`_ALLOWED_PARAMETERS` in
`demo/runner/app.py`) and are mirrored by the dashboard's Pydantic models:
training accepts `data_source`, `delimiter`, `experiment`, `registered_name`,
`dataset_name`, `target`, `alpha`, `l1_ratio`, `test_size`, `random_state`;
batch scoring accepts `data_source`, `delimiter`, `target`, `model_name`,
`model_version`, `experiment`, `chunk_size`, `max_attempts`. Values must be
scalars, because each becomes one command-line flag on the real entrypoint.
A local training trigger looks like this:


```bash
curl -X POST http://localhost:18000/api/runs/train/trigger \
  -H 'content-type: application/json' \
  -H 'X-MS-CLIENT-PRINCIPAL-NAME: demo-user' \
  -d '{"parameters":{"alpha":0.25,"l1_ratio":0.8,"random_state":7}}'
```


The response names the execution and hands back the identifiers to poll:
`execution`, `job_name`, `triggered_by`, the echoed `parameters`, and
`result_id` with a ready-made `result_url`. Locally `result_id` equals the
execution id because the runner exports its execution name as `RESULTS_RUN_ID`,
so one identifier chains from trigger response to results row to
`GET /api/results/{id}`. On ACA the response omits `result_id`; the execution
name addresses the ACA execution API while the outcome lands in the results DB
as usual.

The `X-MS-CLIENT-PRINCIPAL-NAME` header stands in for Entra **Easy Auth**, the
Azure ingress feature that authenticates users and injects the signed-in
identity into that same header. The attribution code is therefore identical in
both environments: locally it says `demo-user`, on Azure it is the caller's real
Entra sign-in name. Unknown or non-scalar parameters are rejected with 422
before anything starts, so a bad request never leaves a misleading running row.


## Promotion semantics

Promotion has no environment-specific machinery. Five rules, all visible in
`src/serving_app/app.py` and `docs/06`:

1. **Images carry code only.** Changing the live model never rebuilds an image;
   model artifacts live in the registry (Blob or MinIO backed), keyed by version.
2. **Models are registry versions.** An evaluated version becomes live through
   the MLflow `production` alias: one atomic flip, `production → N`, against the
   same registry in both backends.
3. **Only the long-running consumer is redeployed.** The serving App is the only
   component holding a model in memory, and it loads once at container startup,
   so promotion repoints its `MODEL_VERSION` and recreates it: a new ACA revision
   on Azure, a Compose recreate locally.
4. **Ephemeral jobs need nothing.** The next batch execution receives the version
   as an ordinary parameter; the runner always passes `--model-version`, and
   scheduled ACA runs read it from the Job definition.
5. **Consumers never poll for new models.** There is no hot reload; the live
   version is exactly what `/readyz` reports, which is what makes it verifiable.

The serving App enforces rule 2 at startup: with `MODEL_VERSION` unset it
refuses to serve rather than resolve a floating alias, so "some recent version"
is not a state the platform can be in.

| Step | Local (Part I) | Azure (Part II) |
|---|---|---|
| Flip the alias | MLflow client call against `http://localhost:<mlflow port>` | same call against the MLflow App URL |
| Repoint the consumer | set `DEMO_MODEL_VERSION` in `demo/.env`, then `docker compose up -d serving` | `az containerapp update --name serving --set-env-vars MODEL_VERSION=N ...` creates a new revision |
| Wrapped by | `python demo/promote.py --backend local --version N` | `python demo/promote.py --backend aca --version N` |
| Rollback | repoint `.env` at the previous N, recreate | another revision pinned at `MODEL_VERSION=N-1`, or revert to the prior revision |

Rollback is the same operation pointed at an older version number. Nothing is
rebuilt on either path because versions and image digests are pointers into the
registry and ACR, not artifacts to recreate.


```bash
curl -X POST http://localhost:18000/api/runs/train/trigger \
  -H 'content-type: application/json' \
  -H 'X-MS-CLIENT-PRINCIPAL-NAME: demo-user' \
  -d '{"parameters":{"alpha":0.25,"l1_ratio":0.8,"random_state":7}}'
```


Under the wrapper, the local backend edits `demo/.env` and shells out to
`docker compose up -d serving`, so only the serving service is recreated; the
ACA backend calls `az containerapp update --set-env-vars MODEL_VERSION=N`,
which rolls a new revision with ACA handling the traffic switch. Either way the
promotion is not finished when the command returns: it is finished when
`GET /readyz` reports exactly the new version, which happens only after the
model is loaded and the startup canary prediction has passed.

This is why `/readyz` exists as a version-aware probe. It turns "which model is
live?" from a question about deployment history into an observable fact, and it
gives the golden-path suite below something exact to assert on.


## Compose ↔ Azure mapping

The full environment mapping, extending the table in `demo/README.md`. The
right-hand column is what Part II declares with **Terraform**, a tool that
provisions cloud resources from declarative configuration files (introduced
properly in chapter 09).

| Platform piece | Part I: Docker Compose | Part II: Azure | Access |
|---|---|---|---|
| Registry + operational DB | Postgres container (`mlflow` and `results` databases) | Azure Database for PostgreSQL | static user/password env vs managed identities with Entra token auth; least-privilege roles from `infra/grants.sql` |
| Object storage | MinIO (S3 API) | Blob Storage | `AWS_*` demo keys vs workload identity; artifact destination `s3://` vs `abfss://` |
| Model registry + tracking UI | MLflow container | MLflow ACA App | Compose DNS name vs HTTPS ingress FQDN |
| Training + batch execution | one-shot Compose services plus the local runner | ACA Jobs (`train`, `batch`) | runner HTTP API vs ACA Jobs API |
| Online serving | serving container | serving ACA App (`id-serving`) | published port vs ingress, probes on `/healthz` and `/readyz` |
| Dashboard | dashboard container, `TRIGGER_BACKEND=local` | dashboard ACA App, `TRIGGER_BACKEND=aca` | plain HTTP vs Easy Auth plus a scoped trigger role |
| Secrets | static demo credentials in the compose YAML | none shared: managed identities and Easy Auth throughout | |

Two rows deserve comment. The local runner has no Azure counterpart by design:
it exists precisely to imitate ACA Job semantics on a laptop, so the dashboard
code is written once against one interaction shape. And the credentials row is
not cosmetic: replacing static passwords with identity is what lets every other
row keep the same application code.


## Behavioral checks, with separate trigger adapters

The contract ends where it can be tested. The local demo/golden_path.py and the
cloud deploy/smoke-tests.sh / .ps1 scripts intentionally use different trigger
mechanisms, but preserve the same assertions:

1. Terminal execution: training and batch reach a terminal success state; cloud
   checks poll ACA execution status, local checks poll the results row.
2. Results state: the batch parent row is SUCCESS after the child work settles.
3. Model identity: promotion targets the production alias, serving /readyz
   reports the exact promoted version, and a prediction response echoes it.
4. Readiness: the serving canary passes before /readyz is accepted.

Only the adapter changes: local trigger calls use the runner and local result IDs;
cloud trigger calls use ACA execution names and the dashboard/results API. Keeping
these suites separate makes failures diagnostic while keeping their behavioral
assertions aligned.
